In [1]:
# Vulnerability Management Machine Learning
# By: John D. Zehnpfennig II
# Created: 12 November 2025
# Description: This app experiments with a variety of ML and AI modules to parse GitLab Ultimate SAST
#              scans, with the intent to learn and predict false positives in new scans.

Initialize the program:

In [2]:
import pandas as pd
import numpy as np
import os, sys
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import tkinter as tk
from tkinter import messagebox, filedialog



Load and train the model:

In [3]:
# --- Model Training Section (same as before) ---
# Load your main CSV for training
df = pd.read_csv("Combined_xlsx.csv")
df['Tool'] = df['Tool'].fillna('Unknown')
df['Scanner Name'] = df['Scanner Name'].fillna('Unknown')
df['Comments'] = df['Comments'].fillna('Unknown')

X = df['Vulnerability']
y_status = df['Status']
y_tool = df['Tool']
y_scanner = df['Scanner Name']
y_comments = df['Comments']

X_train, X_test, y_status_train, y_status_test = train_test_split(X, y_status, test_size=0.2, random_state=42)
_, _, y_tool_train, y_tool_test = train_test_split(X, y_tool, test_size=0.2, random_state=42)
_, _, y_scanner_train, y_scanner_test = train_test_split(X, y_scanner, test_size=0.2, random_state=42)
_, _, y_comments_train, y_comments_test = train_test_split(X, y_comments, test_size=0.2, random_state=42)

vectorizer = TfidfVectorizer(lowercase=True, stop_words='english', max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

status_model = LogisticRegression(max_iter=1000)
status_model.fit(X_train_tfidf, y_status_train)

tool_model = LogisticRegression(max_iter=1000)
tool_model.fit(X_train_tfidf, y_tool_train)

scanner_model = LogisticRegression(max_iter=1000)
scanner_model.fit(X_train_tfidf, y_scanner_train)

comments_model = LogisticRegression(max_iter=1000)
comments_model.fit(X_train_tfidf, y_comments_train)




,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


Initialize and run the GUI

In [4]:
# --- Tkinter GUI Section ---
def predict_vulnerability():
    user_text = entry.get()
    if user_text.lower() == 'x':
        root.destroy()
        return
    if not user_text.strip():
        messagebox.showwarning("Input Error", "Please enter a vulnerability description.")
        return

    new_tfidf = vectorizer.transform([user_text])
    predicted_status = status_model.predict(new_tfidf)[0]
    predicted_tool = tool_model.predict(new_tfidf)[0]
    predicted_scanner = scanner_model.predict(new_tfidf)[0]

    if str(predicted_status).lower() == 'dismissed':
        predicted_comments = comments_model.predict(new_tfidf)[0]
        comments_text = f"Predicted Comments: {predicted_comments}"
    else:
        comments_text = "Predicted Comments: (not applicable)"

    result_label.config(
        text=(
            f"Vulnerability: {user_text}\n"
            f"Predicted Tool: {predicted_tool}\n"
            f"Predicted Scanner Name: {predicted_scanner}\n"
            f"Predicted Status: {predicted_status}\n"
            f"{comments_text}"
        )
    )



Menu and analysis of user or file inputs

In [ ]:
def upload_and_analyze_csv():
    file_path = filedialog.askopenfilename(
        title="Select CSV File",
        filetypes=[("CSV files", "*.csv")]
    )
    if not file_path:
        return

    try:
        df_upload = pd.read_csv(file_path)
    except Exception as e:
        messagebox.showerror("Error", f"Failed to read CSV: {e}")
        return

    if 'Vulnerability' not in df_upload.columns:
        messagebox.showerror("Error", "CSV must contain a 'Vulnerability' column.")
        return

    # Prepare predictions
    vulnerabilities = df_upload['Vulnerability'].fillna("").astype(str)
    tfidf_upload = vectorizer.transform(vulnerabilities)

    df_upload['Predicted_Tool'] = tool_model.predict(tfidf_upload)
    df_upload['Predicted_Scanner_Name'] = scanner_model.predict(tfidf_upload)
    df_upload['Predicted_Status'] = status_model.predict(tfidf_upload)

    # Predict Comments only if Status is 'dismissed'
    predicted_comments = []
    for i, status in enumerate(df_upload['Predicted_Status']):
        if str(status).lower() == 'dismissed':
            comment = comments_model.predict(tfidf_upload[i])
            predicted_comments.append(comment[0])
        else:
            predicted_comments.append('(not applicable)')
    df_upload['Predicted_Comments'] = predicted_comments

    # Save updated CSV
    save_path = filedialog.asksaveasfilename(
        title="Save Updated CSV",
        defaultextension=".csv",
        filetypes=[("CSV files", "*.csv")]
    )
    if save_path:
        df_upload.to_csv(save_path, index=False)
        messagebox.showinfo("Success", f"Updated CSV saved to:\n{save_path}")

root = tk.Tk()
root.title("Vulnerability Triage ML App")

entry_label = tk.Label(root, text="Enter vulnerability description (or 'X' to exit):")
entry_label.pack(pady=5)

entry = tk.Entry(root, width=60)
entry.pack(pady=5)

predict_button = tk.Button(root, text="Predict", command=predict_vulnerability)
predict_button.pack(pady=5)

upload_button = tk.Button(root, text="Upload CSV for Analysis", command=upload_and_analyze_csv)
upload_button.pack(pady=5)

result_label = tk.Label(root, text="", fg="blue")
result_label.pack(pady=10)

root.mainloop()